# 🔍 Interprétabilité — Feature Importance & SHAP
## Projet : Optimisation du ROI Marketing

**Objectif :** Expliquer les décisions du meilleur modèle sélectionné

**Techniques utilisées :**
- Feature Importance native (Random Forest / XGBoost)
- Permutation Importance (agnostique au modèle)
- SHAP — explicabilité globale et locale

---
> Ce notebook répond aux questions métier :
> - Quel canal marketing influence le plus les ventes ?
> - Le type d'influenceur joue-t-il un rôle significatif ?
> - Pourquoi cette combinaison budgétaire génère-t-elle ces ventes ?

## 0. Imports

In [ ]:
import sys, os
sys.path.append('..')
os.makedirs('../reports', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.inspection import permutation_importance

from src.preprocessing.pipeline import (
    load_and_prepare_data, NUM_FEATURES, CAT_FEATURES
)

sns.set_theme(style='whitegrid')
shap.initjs()  # initialise le renderer SHAP
print('✅ Imports OK')

## 1. Chargement des données et du meilleur modèle

In [ ]:
# Données
X_train, X_test, y_train, y_test, preprocessor, feature_names = load_and_prepare_data(
    path='../data/marketing_and_sales.csv'
)

# Chargement de tous les modèles
model_names = ['LinearRegression', 'RandomForest', 'XGBoost', 'MLP']
trained_models = {}
for name in model_names:
    trained_models[name] = joblib.load(f'../models/{name}.pkl')

# Meilleur modèle
best_model = joblib.load('../models/best_model.pkl')
best_name  = [n for n, m in trained_models.items() if type(m) == type(best_model)][0]

print(f'✅ Modèles chargés')
print(f'🏆 Meilleur modèle : {best_name}')
print(f'📋 Features : {feature_names}')

## 2. Feature Importance native (Random Forest & XGBoost)

In [ ]:
tree_models = {k: v for k, v in trained_models.items()
               if k in ['RandomForest', 'XGBoost']}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (name, model) in enumerate(tree_models.items()):
    importances = model.feature_importances_
    sorted_idx  = np.argsort(importances)  # croissant pour barh

    axes[idx].barh(
        [feature_names[i] for i in sorted_idx],
        importances[sorted_idx],
        color='steelblue', edgecolor='white'
    )
    axes[idx].set_title(f'Feature Importance — {name}', fontweight='bold')
    axes[idx].set_xlabel('Importance (réduction impureté)')

plt.suptitle('Feature Importance Native', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/feature_importance_native.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Permutation Importance (agnostique au modèle)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    perm = permutation_importance(
        model, X_test, y_test,
        n_repeats=20,
        random_state=42,
        scoring='r2'
    )

    sorted_idx = perm.importances_mean.argsort()

    axes[idx].barh(
        [feature_names[i] for i in sorted_idx],
        perm.importances_mean[sorted_idx],
        xerr=perm.importances_std[sorted_idx],
        color='coral', edgecolor='white', capsize=3
    )
    axes[idx].axvline(x=0, color='black', linestyle='--', linewidth=1)
    axes[idx].set_title(f'Permutation Importance — {name}', fontweight='bold')
    axes[idx].set_xlabel('Baisse de R² (moyenne ± std)')

plt.suptitle('Permutation Importance — Tous les modèles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP — Explicabilité globale (meilleur modèle)

In [ ]:
print(f'🔄 Calcul des valeurs SHAP pour : {best_name}...')

# Choix de l'explainer selon le type de modèle
if best_name in ['RandomForest', 'XGBoost']:
    explainer   = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test)
else:
    # Pour LinearRegression et MLP
    explainer   = shap.KernelExplainer(best_model.predict, shap.sample(X_train, 50))
    shap_values = explainer.shap_values(X_test[:50])

print('✅ Valeurs SHAP calculées')

In [ ]:
# SHAP Summary Plot (Beeswarm) — importance globale
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test,
    feature_names=feature_names,
    show=False
)
plt.title(f'SHAP Summary Plot — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot — importance moyenne absolue
plt.figure(figsize=(10, 5))
shap.summary_plot(
    shap_values, X_test,
    feature_names=feature_names,
    plot_type='bar',
    show=False
)
plt.title(f'SHAP Feature Importance — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SHAP — Explicabilité locale (prédiction individuelle)

In [ ]:
# Waterfall plot — explication d'une prédiction précise
sample_idx = 0  # ← change cet index pour explorer d'autres campagnes

print(f'📋 Campagne analysée (index {sample_idx}) :')
print(f'   Features       : {dict(zip(feature_names, X_test[sample_idx].round(3)))}')
print(f'   Ventes réelles : {y_test[sample_idx]:.2f} M')
print(f'   Ventes prédites: {best_model.predict(X_test[sample_idx:sample_idx+1])[0]:.2f} M')

if best_name in ['RandomForest', 'XGBoost']:
    shap_exp = shap.Explanation(
        values        = shap_values[sample_idx],
        base_values   = explainer.expected_value,
        data          = X_test[sample_idx],
        feature_names = feature_names
    )
    plt.figure(figsize=(10, 5))
    shap.plots.waterfall(shap_exp, show=False)
    plt.title(f'SHAP Waterfall — Campagne #{sample_idx}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('ℹ️ Waterfall disponible uniquement pour les modèles basés sur les arbres.')

## 6. SHAP Dependence Plot — effet d'un canal sur les ventes

In [ ]:
if best_name in ['RandomForest', 'XGBoost']:
    # Top feature (la plus importante)
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_feature_idx  = mean_abs_shap.argmax()
    top_feature_name = feature_names[top_feature_idx]

    fig, axes = plt.subplots(1, len(NUM_FEATURES), figsize=(6 * len(NUM_FEATURES), 5))

    for i, feat in enumerate(NUM_FEATURES):
        feat_idx = feature_names.index(feat)
        ax = axes[i] if len(NUM_FEATURES) > 1 else axes
        shap.dependence_plot(
            feat_idx, shap_values, X_test,
            feature_names=feature_names,
            ax=ax, show=False
        )
        ax.set_title(f'SHAP Dependence — {feat}', fontweight='bold')

    plt.suptitle('Impact de chaque canal sur les ventes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/shap_dependence.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('ℹ️ Dependence plot disponible uniquement pour les modèles basés sur les arbres.')

## 7. Synthèse — Insights métier

In [ ]:
# Ranking des features par importance SHAP moyenne
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_ranking  = pd.DataFrame({
    'Feature'         : feature_names,
    'SHAP Importance' : mean_abs_shap.round(4)
}).sort_values('SHAP Importance', ascending=False).reset_index(drop=True)

print('=== RANKING SHAP — Variables les plus influentes ===')
display(shap_ranking)

shap_ranking.to_csv('../reports/shap_ranking.csv', index=False)
print('\n💾 Ranking sauvegardé → reports/shap_ranking.csv')

print("""
╔══════════════════════════════════════════════════════════╗
║           INSIGHTS MÉTIER — À COMPLÉTER                 ║
╠══════════════════════════════════════════════════════════╣

📢 Canal le plus influent  : [compléter selon tes résultats]
📢 Canal le moins influent : [compléter]
📢 Rôle du type d'influenceur : [significatif / marginal ?]

💡 Recommandations business :
  → Augmenter le budget [X] pour maximiser les ventes
  → Le canal [Y] montre un rendement marginal décroissant
  → Privilégier les influenceurs de type [Z] pour ce budget

🚀 Prochaines étapes :
  → api/main.py        : exposition via FastAPI
  → dashboard/app.py   : interface décisionnelle Streamlit
""")